# Next-Day AQI Prediction for Indian Cities (KDD approach)
**How to run:** set `AUTHOR_NAME` in the first code cell, then *Runtime -> Run all*. The dataset downloads automatically; if that fails you will be asked to upload `city_day.csv`.

In [ ]:
# Step 0: install libraries (about 1 minute on Colab)
!pip -q install xgboost shap python-pptx kagglehub

In [ ]:
"""
Next-Day AQI Prediction for Indian Cities  --  KDD-based pipeline
==================================================================
Data Selection -> EDA -> Preprocessing -> Feature Engineering -> Chronological Split
-> Baselines -> Advanced Models (RF, XGBoost) -> AQI Classification -> Evaluation -> SHAP

USAGE
  1. pip install -r requirements.txt
  2. Edit the CONFIG block below (at least AUTHOR_NAME).
  3. From the repo root run:  python src/aqi_kdd_pipeline.py
     (the dataset is downloaded to data/city_day.csv automatically;
      or download it yourself from
      https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india)

OUTPUT (in ./outputs): figures/*.png, results.json, AQI_Prediction_KDD.pptx
"""
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             f1_score, accuracy_score, confusion_matrix)
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor, XGBClassifier

warnings.filterwarnings("ignore")

# ============================== CONFIG ==============================
DATA_PATH   = os.environ.get("AQI_DATA", "data/city_day.csv")
OUT_DIR     = "outputs"
AUTHOR_NAME = "Your Name"            # <-- put your name here
SUBTITLE    = "KDD Mini Project"     # e.g. course / department / institute
GUIDE       = ""                     # optional, e.g. "Guided by: Prof. XYZ"
SEED        = 42
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15    # remaining 15% = test (chronological)
MISSING_DROP_THRESHOLD = 0.50        # drop pollutant columns missing more than this
# ====================================================================

def ensure_data(path):
    """Use data/city_day.csv if present, else try to download it with kagglehub."""
    if os.path.exists(path):
        return
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    try:
        import glob, shutil, kagglehub
        folder = kagglehub.dataset_download("rohanrao/air-quality-data-in-india")
        shutil.copy(glob.glob(os.path.join(folder, "**", "city_day.csv"), recursive=True)[0], path)
        print("Downloaded dataset to", path)
    except Exception as e:
        raise FileNotFoundError(
            f"{path} not found and auto-download failed ({type(e).__name__}). Download city_day.csv from "
            "https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india and place it in the data/ folder.")

FIG = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG, exist_ok=True)
np.random.seed(SEED)

CATS = ["Good", "Satisfactory", "Moderate", "Poor", "Very Poor", "Severe"]
def aqi_to_cat(a):
    """CPCB AQI buckets -> integer 0..5"""
    return np.digitize(a, [50, 100, 200, 300, 400])  # 0..5

def savefig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(FIG, name), dpi=160, bbox_inches="tight")
    plt.show()

In [ ]:
# Step 0b: get the dataset -> data/city_day.csv
import os, glob, shutil
os.makedirs(os.path.dirname(DATA_PATH) or ".", exist_ok=True)
if not os.path.exists(DATA_PATH):
    try:                                   # Option A: automatic download from Kaggle
        import kagglehub
        folder = kagglehub.dataset_download("rohanrao/air-quality-data-in-india")
        found = glob.glob(os.path.join(folder, "**", "city_day.csv"), recursive=True)
        shutil.copy(found[0], DATA_PATH)
        print("Downloaded from Kaggle")
    except Exception as e:                 # Option B: upload the file yourself (Colab)
        print("Auto-download failed (", type(e).__name__, "). Upload city_day.csv now...")
        from google.colab import files
        up = files.upload()
        for name in up:
            if name.lower().endswith(".csv"):
                shutil.move(name, DATA_PATH); break
assert os.path.exists(DATA_PATH), "city_day.csv not found - upload it into the data/ folder."
print("Data ready:", DATA_PATH, round(os.path.getsize(DATA_PATH)/1e6, 1), "MB")

In [ ]:
# ------------------------------------------------------------------
# 1. DATA SELECTION
# ------------------------------------------------------------------
raw = pd.read_csv(DATA_PATH, parse_dates=["Date"])
raw = raw.rename(columns={"PM2.5": "PM25"})
POLLUTANTS_ALL = ["PM25", "PM10", "NO", "NO2", "NOx", "NH3", "CO", "SO2", "O3",
                  "Benzene", "Toluene", "Xylene"]
POLLUTANTS_ALL = [c for c in POLLUTANTS_ALL if c in raw.columns]
info = dict(
    n_rows=int(len(raw)), n_cities=int(raw["City"].nunique()),
    date_min=str(raw["Date"].min().date()), date_max=str(raw["Date"].max().date()),
    n_aqi_missing=int(raw["AQI"].isna().sum()),
)
miss_pct = raw[POLLUTANTS_ALL + ["AQI"]].isna().mean().sort_values(ascending=False) * 100
info["missing_pct"] = {k: round(v, 1) for k, v in miss_pct.items()}
print("Selected data:", info["n_rows"], "rows,", info["n_cities"], "cities")

In [ ]:
# ------------------------------------------------------------------
# 2. EDA
# ------------------------------------------------------------------
plt.figure(figsize=(7, 4))
miss_pct.sort_values().plot.barh(color="#1f9e9e")
plt.xlabel("% missing"); plt.title("Missing values per column")
savefig("eda_missing.png")

plt.figure(figsize=(7, 4))
raw["AQI"].dropna().plot.hist(bins=60, color="#1f9e9e", edgecolor="white")
plt.xlabel("AQI"); plt.title("AQI distribution (right-skewed)")
savefig("eda_aqi_dist.png")

order = raw.groupby("City")["AQI"].median().sort_values(ascending=False).index
plt.figure(figsize=(9, 4.5))
plt.boxplot([raw.loc[raw.City == c, "AQI"].dropna() for c in order],
            labels=order, showfliers=False, patch_artist=True,
            boxprops=dict(facecolor="#9fd6d6"))
plt.xticks(rotation=70, fontsize=8); plt.ylabel("AQI"); plt.title("AQI by city")
savefig("eda_city_box.png")

m = raw.dropna(subset=["AQI"]).assign(month=lambda d: d.Date.dt.month)
plt.figure(figsize=(7, 4))
m.groupby("month")["AQI"].mean().plot(marker="o", color="#d1495b")
plt.xticks(range(1, 13)); plt.ylabel("Mean AQI")
plt.title("Seasonality: winter peak, monsoon trough")
savefig("eda_monthly.png")

plt.figure(figsize=(6.5, 5.5))
corr = raw[POLLUTANTS_ALL + ["AQI"]].corr()
plt.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
plt.xticks(range(len(corr)), corr.columns, rotation=75, fontsize=8)
plt.yticks(range(len(corr)), corr.columns, fontsize=8)
plt.colorbar(shrink=0.8); plt.title("Correlation matrix")
savefig("eda_corr.png")

info["corr_with_aqi"] = corr["AQI"].drop("AQI").sort_values(ascending=False).round(2).to_dict()

In [ ]:
# ------------------------------------------------------------------
# 3. PREPROCESSING
# ------------------------------------------------------------------
keep_poll = [c for c in POLLUTANTS_ALL if miss_pct.get(c, 0) / 100 <= MISSING_DROP_THRESHOLD]
dropped_cols = [c for c in POLLUTANTS_ALL if c not in keep_poll]
print("Dropped (too many missing):", dropped_cols)

# regularise each city to a full daily calendar so that shift() == "previous/next day"
parts = []
for city, g in raw.groupby("City"):
    g = g.drop_duplicates("Date").set_index("Date").sort_index().asfreq("D")
    g["City"] = city
    parts.append(g.reset_index())
d = pd.concat(parts, ignore_index=True)
d = d.rename(columns={"AQI": "AQI_raw"})
d["AQI_f"] = d["AQI_raw"]
# causal gap filling: forward-fill at most 3 days (never uses future values)
fill_cols = keep_poll + ["AQI_f"]
d[fill_cols] = d.groupby("City")[fill_cols].transform(lambda s: s.ffill(limit=3))

In [ ]:
# ------------------------------------------------------------------
# 4. FEATURE ENGINEERING
# ------------------------------------------------------------------
g = d.groupby("City")
for k in (1, 2, 3, 7):
    for c in ("AQI_f", "PM25", "PM10"):
        if c in d: d[f"{c}_lag{k}"] = g[c].shift(k)
for w in (3, 7):
    for c in ("AQI_f", "PM25"):
        if c in d:
            d[f"{c}_roll{w}"] = g[c].transform(lambda s, w=w: s.rolling(w, min_periods=max(2, w // 2)).mean())
d["AQI_f_std7"] = g["AQI_f"].transform(lambda s: s.rolling(7, min_periods=4).std())
d["AQI_f_diff1"] = d["AQI_f"] - d["AQI_f_lag1"]
d["month"] = d["Date"].dt.month
d["dow"] = d["Date"].dt.dayofweek
doy = d["Date"].dt.dayofyear
d["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
d["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)
d["is_winter"] = d["month"].isin([11, 12, 1, 2]).astype(int)
d["target_AQI"] = g["AQI_raw"].shift(-1)                  # next-day AQI (observed only)
d["target_CAT"] = aqi_to_cat(d["target_AQI"].fillna(-1)).astype(int)
d.loc[d["target_AQI"].isna(), "target_CAT"] = -1
d = d.rename(columns={"AQI_f": "AQI_t"}).rename(
    columns=lambda c: c.replace("AQI_f_", "AQI_"))          # tidy names

city_dummies = pd.get_dummies(d["City"], prefix="city", dtype=float)
d = pd.concat([d, city_dummies], axis=1)

must_have = ["AQI_t", "AQI_lag1", "AQI_lag7", "target_AQI"]
d = d.dropna(subset=must_have).reset_index(drop=True)

FEATURES = (keep_poll + ["AQI_t"]
            + [c for c in d.columns if "_lag" in c or "_roll" in c]
            + ["AQI_std7", "AQI_diff1", "month", "dow", "doy_sin", "doy_cos", "is_winter"]
            + list(city_dummies.columns))
FEATURES = list(dict.fromkeys(FEATURES))
print("Model rows:", len(d), " Features:", len(FEATURES))

In [ ]:
# ------------------------------------------------------------------
# 5. CHRONOLOGICAL SPLIT  (no shuffling -> no look-ahead leakage)
# ------------------------------------------------------------------
dates = np.sort(d["Date"].unique())
t_end = dates[int(len(dates) * TRAIN_FRAC)]
v_end = dates[int(len(dates) * (TRAIN_FRAC + VAL_FRAC))]
tr = d[d.Date < t_end].copy()
va = d[(d.Date >= t_end) & (d.Date < v_end)].copy()
te = d[d.Date >= v_end].copy()
split_info = {
    "train": [str(tr.Date.min().date()), str(tr.Date.max().date()), len(tr)],
    "val":   [str(va.Date.min().date()), str(va.Date.max().date()), len(va)],
    "test":  [str(te.Date.min().date()), str(te.Date.max().date()), len(te)],
}
print(split_info)

# train-only imputation + outlier capping (fit on train, apply everywhere)
med = tr[FEATURES].median()
cap_cols = keep_poll
hi = tr[cap_cols].quantile(0.995)
for part in (tr, va, te):
    part[FEATURES] = part[FEATURES].fillna(med)
    part[cap_cols] = part[cap_cols].clip(upper=hi, axis=1)

Xtr, Xva, Xte = tr[FEATURES], va[FEATURES], te[FEATURES]
ytr, yva, yte = tr.target_AQI.values, va.target_AQI.values, te.target_AQI.values
ctr, cva, cte = tr.target_CAT.values, va.target_CAT.values, te.target_CAT.values

In [ ]:
# ------------------------------------------------------------------
# 6/7. REGRESSION: BASELINES + ADVANCED MODELS
# ------------------------------------------------------------------
def reg_metrics(y, p):
    return dict(MAE=float(mean_absolute_error(y, p)),
                RMSE=float(np.sqrt(mean_squared_error(y, p))),
                R2=float(r2_score(y, p)))

reg_models, reg_pred = {}, {}
reg_pred["Persistence (AQI today)"] = Xte["AQI_t"].values
lin = make_pipeline(StandardScaler(), LinearRegression()).fit(Xtr, ytr)
rdg = make_pipeline(StandardScaler(), Ridge(alpha=10)).fit(Xtr, ytr)
rf = RandomForestRegressor(n_estimators=300, max_depth=18, min_samples_leaf=3,
                           n_jobs=-1, random_state=SEED).fit(Xtr, ytr)
xgb = XGBRegressor(n_estimators=1000, learning_rate=0.03, max_depth=6, subsample=0.8,
                   colsample_bytree=0.8, min_child_weight=3, reg_lambda=2,
                   early_stopping_rounds=50, random_state=SEED, n_jobs=-1)
xgb.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
for name, mdl in [("Linear Regression", lin), ("Ridge Regression", rdg),
                  ("Random Forest", rf), ("XGBoost", xgb)]:
    reg_models[name] = mdl
    reg_pred[name] = mdl.predict(Xte)

reg_results = {n: reg_metrics(yte, p) for n, p in reg_pred.items()}
print(pd.DataFrame(reg_results).T.round(3))
best_reg = min(reg_results, key=lambda n: reg_results[n]["RMSE"])

In [ ]:
# ------------------------------------------------------------------
# 8. CLASSIFICATION (AQI category)
# ------------------------------------------------------------------
present = sorted(set(ctr))
to_idx = {c: i for i, c in enumerate(present)}
from_idx = np.array(present)

def clf_metrics(y, p):
    return dict(Accuracy=float(accuracy_score(y, p)),
                F1_macro=float(f1_score(y, p, average="macro", zero_division=0)),
                F1_weighted=float(f1_score(y, p, average="weighted", zero_division=0)))

clf_pred = {}
clf_pred["Persistence (category today)"] = aqi_to_cat(Xte["AQI_t"].values)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")).fit(Xtr, ctr)
rfc = RandomForestClassifier(n_estimators=300, max_depth=18, min_samples_leaf=3,
                             class_weight="balanced", n_jobs=-1, random_state=SEED).fit(Xtr, ctr)
sw = compute_sample_weight("balanced", ctr)
xgc = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=5, subsample=0.8,
                    colsample_bytree=0.8, random_state=SEED, n_jobs=-1)
xgc.fit(Xtr, np.vectorize(to_idx.get)(ctr), sample_weight=sw)
clf_pred["Logistic Regression"] = lr.predict(Xte)
clf_pred["Random Forest"] = rfc.predict(Xte)
clf_pred["XGBoost"] = from_idx[xgc.predict(Xte)]
clf_pred["XGBoost regressor -> binned"] = aqi_to_cat(reg_pred["XGBoost"])

clf_results = {n: clf_metrics(cte, p) for n, p in clf_pred.items()}
print(pd.DataFrame(clf_results).T.round(3))
best_clf = max(clf_results, key=lambda n: clf_results[n]["F1_macro"])

In [ ]:
# ------------------------------------------------------------------
# 9. EVALUATION FIGURES

In [ ]:
# ------------------------------------------------------------------
# model comparison
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
names = list(reg_results)
ax[0].barh(names, [reg_results[n]["RMSE"] for n in names], color="#1f9e9e")
ax[0].set_title("Test RMSE (lower is better)"); ax[0].invert_yaxis()
cn = list(clf_results)
ax[1].barh(cn, [clf_results[n]["F1_macro"] for n in cn], color="#d1495b")
ax[1].set_title("Test macro-F1 (higher is better)"); ax[1].invert_yaxis()
savefig("model_comparison.png")

# predicted vs actual
bp = reg_pred[best_reg]
plt.figure(figsize=(5.5, 5))
plt.scatter(yte, bp, s=6, alpha=0.35, color="#1f9e9e")
lim = [0, max(yte.max(), bp.max())]
plt.plot(lim, lim, "k--", lw=1)
plt.xlabel("Actual next-day AQI"); plt.ylabel("Predicted"); plt.title(f"{best_reg}: predicted vs actual")
savefig("pred_vs_actual.png")

# time series for the most polluted city in the test window
city_show = te.groupby("City")["target_AQI"].mean().idxmax()
msk = (te.City == city_show).values
plt.figure(figsize=(9, 3.8))
plt.plot(te.Date[msk].values, yte[msk], label="Actual", color="#333", lw=1.2)
plt.plot(te.Date[msk].values, bp[msk], label=best_reg, color="#d1495b", lw=1.2)
plt.title(f"{city_show}: next-day AQI, test period"); plt.legend(); plt.ylabel("AQI")
savefig("timeseries_test.png")

# confusion matrix for best classifier
labels = list(range(6))
cm = confusion_matrix(cte, clf_pred[best_clf], labels=labels)
cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
plt.figure(figsize=(6, 5))
plt.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
plt.xticks(labels, CATS, rotation=45, ha="right", fontsize=8); plt.yticks(labels, CATS, fontsize=8)
for i in labels:
    for j in labels:
        if cm[i, j]: plt.text(j, i, cm[i, j], ha="center", va="center",
                              color="white" if cmn[i, j] > 0.5 else "black", fontsize=8)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"Confusion matrix: {best_clf}")
savefig("confusion_matrix.png")

In [ ]:
# ------------------------------------------------------------------
# 10. INTERPRETATION (SHAP)
# ------------------------------------------------------------------
top_feats = []
try:
    import shap
    sample = Xte.sample(min(2000, len(Xte)), random_state=SEED)
    expl = shap.TreeExplainer(xgb)
    sv = expl.shap_values(sample)
    plt.figure(); shap.summary_plot(sv, sample, max_display=15, show=False)
    plt.title("SHAP summary: XGBoost next-day AQI"); savefig("shap_summary.png")
    plt.figure(); shap.summary_plot(sv, sample, plot_type="bar", max_display=15, show=False)
    plt.title("Mean |SHAP|"); savefig("shap_bar.png")
    imp = pd.Series(np.abs(sv).mean(0), index=sample.columns).sort_values(ascending=False)
    interp_method = "SHAP"
except Exception as e:                       # fallback if shap is unavailable
    print("SHAP failed, using permutation importance:", e)
    pi = permutation_importance(xgb, Xte, yte, n_repeats=5, random_state=SEED, n_jobs=-1)
    imp = pd.Series(pi.importances_mean, index=Xte.columns).sort_values(ascending=False)
    plt.figure(figsize=(6, 5)); imp.head(15)[::-1].plot.barh(color="#1f9e9e")
    plt.title("Permutation importance"); savefig("shap_bar.png")
    interp_method = "Permutation importance"
top_feats = [(k, float(v)) for k, v in imp.head(10).items()]

In [ ]:
# ------------------------------------------------------------------
# SAVE RESULTS
# ------------------------------------------------------------------
results = dict(info=info, dropped_cols=dropped_cols, n_features=len(FEATURES),
               n_model_rows=int(len(d)), split=split_info, reg=reg_results, clf=clf_results,
               best_reg=best_reg, best_clf=best_clf, top_features=top_feats,
               interp_method=interp_method, city_show=city_show)
with open(os.path.join(OUT_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2, default=str)

In [ ]:
# ------------------------------------------------------------------
# BUILD THE PPT  (python-pptx)
# ------------------------------------------------------------------
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.shapes import MSO_SHAPE
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from PIL import Image

NAVY, TEAL, RED = RGBColor(0x0B, 0x25, 0x45), RGBColor(0x1F, 0x9E, 0x9E), RGBColor(0xD1, 0x49, 0x5B)
INK, MUTE, LIGHT, WHITE = RGBColor(0x22, 0x2B, 0x38), RGBColor(0x6B, 0x75, 0x84), RGBColor(0xEE, 0xF4, 0xF6), RGBColor(255, 255, 255)

prs = Presentation()
prs.slide_width, prs.slide_height = Inches(13.333), Inches(7.5)
BLANK = prs.slide_layouts[6]
page = {"n": 0}

def rect(s, x, y, w, h, color, shape=MSO_SHAPE.RECTANGLE):
    r = s.shapes.add_shape(shape, Inches(x), Inches(y), Inches(w), Inches(h))
    r.fill.solid(); r.fill.fore_color.rgb = color; r.line.fill.background()
    return r

def text(s, t, x, y, w, h, size=18, bold=False, color=INK, align=PP_ALIGN.LEFT, anchor=MSO_ANCHOR.TOP):
    tb = s.shapes.add_textbox(Inches(x), Inches(y), Inches(w), Inches(h))
    tf = tb.text_frame; tf.word_wrap = True; tf.vertical_anchor = anchor
    tf.margin_left = tf.margin_right = 0
    p = tf.paragraphs[0]; p.alignment = align
    r = p.add_run(); r.text = t
    r.font.size, r.font.bold, r.font.color.rgb, r.font.name = Pt(size), bold, color, "Calibri"
    return tb

def bullets(s, items, x, y, w, h, size=18):
    tb = s.shapes.add_textbox(Inches(x), Inches(y), Inches(w), Inches(h))
    tf = tb.text_frame; tf.word_wrap = True; tf.margin_left = tf.margin_right = 0
    for i, it in enumerate(items):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.space_after = Pt(10)
        r = p.add_run(); r.text = "\u2022  " + it
        r.font.size, r.font.color.rgb, r.font.name = Pt(size), INK, "Calibri"

def slide(title, section=None):
    s = prs.slides.add_slide(BLANK); page["n"] += 1
    rect(s, 0, 0, 13.333, 1.1, NAVY)
    rect(s, 0, 1.1, 13.333, 0.06, TEAL)
    text(s, title, 0.6, 0.15, 11, 0.8, 28, True, WHITE, anchor=MSO_ANCHOR.MIDDLE)
    if section:
        text(s, section, 10.3, 0.15, 2.5, 0.8, 12, False, RGBColor(0x9F, 0xD6, 0xD6),
             PP_ALIGN.RIGHT, MSO_ANCHOR.MIDDLE)
    text(s, f"{AUTHOR_NAME}  |  Next-Day AQI Prediction", 0.6, 7.05, 8, 0.3, 10, False, MUTE)
    if os.environ.get("SAMPLE_BANNER") == "1":   # only used for the preview deck built on synthetic data
        text(s, "SAMPLE DATA - NOT REAL RESULTS", 5.0, 7.05, 4, 0.3, 11, True, RED, PP_ALIGN.CENTER)
    text(s, str(page["n"]), 12.2, 7.05, 0.6, 0.3, 10, False, MUTE, PP_ALIGN.RIGHT)
    return s

def picture(s, name, x, y, w, h):
    path = os.path.join(FIG, name)
    if not os.path.exists(path): return
    iw, ih = Image.open(path).size
    sc = min(w / iw, h / ih)
    pw, ph = iw * sc, ih * sc
    s.shapes.add_picture(path, Inches(x + (w - pw) / 2), Inches(y + (h - ph) / 2), Inches(pw), Inches(ph))

def table(s, header, rows, x, y, w, col_w=None, size=13, hl_row=None):
    rh = 0.42
    gt = s.shapes.add_table(len(rows) + 1, len(header), Inches(x), Inches(y), Inches(w), Inches(rh * (len(rows) + 1)))
    t = gt.table
    if col_w:
        for i, cw in enumerate(col_w): t.columns[i].width = Inches(cw)
    for j, h in enumerate(header):
        c = t.cell(0, j); c.fill.solid(); c.fill.fore_color.rgb = NAVY
        c.text_frame.text = h
        r = c.text_frame.paragraphs[0].runs[0]; r.font.size, r.font.bold, r.font.color.rgb = Pt(size), True, WHITE
    for i, row in enumerate(rows, 1):
        for j, v in enumerate(row):
            c = t.cell(i, j); c.fill.solid()
            c.fill.fore_color.rgb = RGBColor(0xD9, 0xF0, 0xEF) if hl_row == i - 1 else (LIGHT if i % 2 else WHITE)
            c.text_frame.text = str(v)
            para = c.text_frame.paragraphs[0]
            para.alignment = PP_ALIGN.LEFT if j == 0 else PP_ALIGN.CENTER
            r = para.runs[0]; r.font.size, r.font.color.rgb = Pt(size), INK
            r.font.bold = (hl_row == i - 1)

def stat(s, x, y, w, big, small, color=TEAL):
    rect(s, x, y, w, 1.5, LIGHT)
    rect(s, x, y, 0.08, 1.5, color)
    text(s, big, x + 0.3, y + 0.15, w - 0.4, 0.8, 32, True, color)
    text(s, small, x + 0.3, y + 0.92, w - 0.4, 0.5, 13, False, MUTE)

R = results
b_reg, b_clf = R["best_reg"], R["best_clf"]
pers = R["reg"]["Persistence (AQI today)"]
bm = R["reg"][b_reg]
gain = (pers["RMSE"] - bm["RMSE"]) / pers["RMSE"] * 100

# 1 Title
s = prs.slides.add_slide(BLANK); page["n"] += 1
rect(s, 0, 0, 13.333, 7.5, NAVY); rect(s, 0.8, 3.55, 1.6, 0.07, TEAL)
text(s, "Next-Day Air Quality Index (AQI) Prediction for Indian Cities", 0.8, 1.2, 11.5, 1.6, 40, True, WHITE)
text(s, "Using Machine Learning: A KDD-Based Approach", 0.8, 2.7, 11.5, 0.7, 24, False, RGBColor(0x9F, 0xD6, 0xD6))
text(s, AUTHOR_NAME, 0.8, 4.0, 11, 0.6, 26, True, WHITE)
text(s, SUBTITLE + (("   |   " + GUIDE) if GUIDE else ""), 0.8, 4.65, 11, 0.5, 16, False, RGBColor(0xC8, 0xD3, 0xDF))

# 2 Problem
s = slide("Problem Statement & Objectives", "Introduction")
bullets(s, ["Air pollution in Indian cities harms public health; timely forecasts enable advisories and restrictions",
            "Goal: use a city's current and recent pollutant readings to predict tomorrow's AQI",
            "Regression: next-day AQI value   |   Classification: next-day AQI category (6 CPCB classes)",
            "Apply every KDD stage to real, messy data and compare baseline vs advanced models",
            "Identify which pollutants and features drive the forecast"], 0.6, 1.6, 12, 5, 20)

# 3 KDD
s = slide("KDD Process Followed", "Methodology")
steps = ["Data\nSelection", "EDA", "Pre-\nprocessing", "Feature\nEngineering", "Chrono.\nSplit", "Modelling", "Evaluation", "SHAP"]
bw = 1.35
for i, t_ in enumerate(steps):
    x = 0.6 + i * (bw + 0.22)
    r = rect(s, x, 2.7, bw, 1.4, TEAL if i % 2 == 0 else NAVY, MSO_SHAPE.ROUNDED_RECTANGLE)
    tf_ = r.text_frame; tf_.word_wrap = True; tf_.vertical_anchor = MSO_ANCHOR.MIDDLE
    tf_.text = t_.replace("\n", " ")
    for para in tf_.paragraphs:
        para.alignment = PP_ALIGN.CENTER
        for run in para.runs:
            run.font.size, run.font.bold, run.font.color.rgb = Pt(14), True, WHITE
text(s, "Models: Persistence, Linear, Ridge, Random Forest, XGBoost (regression) and "
        "Logistic, Random Forest, XGBoost (classification)", 0.6, 4.7, 12, 1, 18, False, INK)

# 4 Data selection
s = slide("Data Selection", "KDD 1")
stat(s, 0.6, 1.6, 2.9, f"{R['info']['n_rows']:,}", "daily city records")
stat(s, 3.7, 1.6, 2.9, str(R["info"]["n_cities"]), "Indian cities", RED)
stat(s, 6.8, 1.6, 2.9, f"{R['info']['date_min'][:4]}\u2013{R['info']['date_max'][:4]}", "date coverage")
stat(s, 9.9, 1.6, 2.9, f"{R['info']['n_aqi_missing']:,}", "rows with AQI missing", RED)
bullets(s, ["Source: Kaggle \u2013 Air Quality Data in India (city_day.csv, CPCB stations)",
            "Features: PM2.5, PM10, NO, NO2, NOx, NH3, CO, SO2, O3, Benzene, Toluene, Xylene",
            "Targets: next-day AQI (regression) and next-day AQI bucket (classification)",
            "Rows without an observed AQI cannot serve as target and are excluded as such"], 0.6, 3.6, 12, 3, 18)

# 5 EDA 1
s = slide("EDA: Distribution & City Differences", "KDD 2")
picture(s, "eda_aqi_dist.png", 0.5, 1.4, 5.2, 4.2)
picture(s, "eda_city_box.png", 5.8, 1.4, 7.1, 4.2)
top3 = ", ".join(order[:3])
text(s, f"AQI is right-skewed with a long tail of severe days; most polluted cities by median AQI: {top3}.",
     0.6, 5.9, 12, 0.9, 16, False, INK)

# 6 EDA 2
s = slide("EDA: Missing Data, Seasonality & Correlation", "KDD 2")
picture(s, "eda_missing.png", 0.4, 1.35, 4.3, 3.2)
picture(s, "eda_monthly.png", 4.7, 1.35, 4.3, 3.2)
picture(s, "eda_corr.png", 9.0, 1.3, 4.0, 3.6)
cw = list(R["info"]["corr_with_aqi"].items())[:3]
bullets(s, ["Several pollutant columns have substantial gaps \u2192 imputation / column dropping needed",
            "Clear seasonal pattern: pollution rises in winter, falls in the monsoon",
            "Strongest correlates of AQI: " + ", ".join(f"{k} ({v})" for k, v in cw)], 0.6, 5.0, 12, 2, 16)

# 7 Preprocessing
s = slide("Preprocessing", "KDD 3")
dc = ", ".join(R["dropped_cols"]) if R["dropped_cols"] else "none"
bullets(s, [f"Columns with > {int(MISSING_DROP_THRESHOLD*100)}% missing dropped: {dc}",
            "Each city reindexed to a full daily calendar so 'previous / next day' is exact",
            "Gaps filled by forward-fill (max 3 days) \u2013 causal, never uses future values",
            "Remaining gaps imputed with TRAIN-set medians only (no leakage)",
            "Extreme outliers capped at the train 99.5th percentile",
            "Rows lacking an observed next-day AQI removed from modelling"], 0.6, 1.6, 12, 5, 19)

# 8 Feature engineering
s = slide("Feature Engineering", "KDD 4")
bullets(s, ["Lags (1, 2, 3, 7 days) of AQI, PM2.5, PM10",
            "Rolling mean (3 and 7 days) of AQI and PM2.5; rolling std and day-to-day change of AQI",
            "Calendar: month, day of week, sin/cos of day-of-year, winter flag",
            "City one-hot encoding to capture local baselines",
            f"Total: {R['n_features']} features on {R['n_model_rows']:,} modelling rows",
            "Target = AQI of the next day; category from CPCB bins (0\u201350, 51\u2013100, 101\u2013200, 201\u2013300, 301\u2013400, 400+)"],
        0.6, 1.6, 12, 5, 19)

# 9 Split
s = slide("Chronological Train / Validation / Test Split", "KDD 5")
table(s, ["Set", "From", "To", "Rows"],
      [[k.title(), v[0], v[1], f"{v[2]:,}"] for k, v in R["split"].items()],
      0.6, 1.7, 8.5, [2, 2.5, 2.5, 1.5], 16)
bullets(s, ["Split by date, never shuffled: the model only ever predicts the future",
            "Validation set used for XGBoost early stopping",
            "Test set touched once for final reporting",
            "The test window may include the 2020 COVID-19 lockdown, a distribution shift"],
        0.6, 3.9, 12, 3, 18)

# 10 Regression results
s = slide("Regression Results (Test Set)", "Evaluation")
rows = [[n, f"{m['MAE']:.2f}", f"{m['RMSE']:.2f}", f"{m['R2']:.3f}"] for n, m in R["reg"].items()]
table(s, ["Model", "MAE", "RMSE", "R\u00b2"], rows, 0.6, 1.6, 7.2, [3.3, 1.3, 1.3, 1.3], 15,
      hl_row=list(R["reg"]).index(b_reg))
stat(s, 8.4, 1.6, 4.3, b_reg, "best model by test RMSE")
stat(s, 8.4, 3.35, 4.3, f"{gain:.1f}%", "RMSE reduction vs persistence baseline", RED)
picture(s, "model_comparison.png", 0.6, 4.3, 7.2, 2.6)

# 11 Pred vs actual
s = slide("Predicted vs Actual Next-Day AQI", "Evaluation")
picture(s, "pred_vs_actual.png", 0.4, 1.35, 4.8, 4.8)
picture(s, "timeseries_test.png", 5.3, 1.5, 7.7, 3.4)
text(s, f"{b_reg}: MAE {bm['MAE']:.1f}, RMSE {bm['RMSE']:.1f}, R\u00b2 {bm['R2']:.2f}. "
        "See the scatter (closer to the diagonal is better) and the test-period time series.",
     5.4, 5.1, 7.5, 1.4, 16, False, INK)

# 12 Classification
s = slide("AQI Category Classification (Test Set)", "Evaluation")
rows = [[n, f"{m['Accuracy']:.3f}", f"{m['F1_macro']:.3f}", f"{m['F1_weighted']:.3f}"] for n, m in R["clf"].items()]
table(s, ["Model", "Accuracy", "F1 macro", "F1 weighted"], rows, 0.5, 1.6, 7.6, [3.4, 1.4, 1.4, 1.4], 13,
      hl_row=list(R["clf"]).index(b_clf))
picture(s, "confusion_matrix.png", 8.3, 1.4, 4.7, 4.2)
text(s, f"Best by macro-F1: {b_clf}. Macro-F1 is used because classes are imbalanced; "
        "read the confusion matrix for which categories get confused.", 0.6, 5.0, 7.5, 1.5, 16, False, INK)

# 13 SHAP
s = slide(f"Interpretation: What Drives the Forecast ({R['interp_method']})", "Interpretation")
picture(s, "shap_summary.png" if os.path.exists(os.path.join(FIG, "shap_summary.png")) else "shap_bar.png",
        0.4, 1.35, 7.2, 5.5)
tf = [f for f, _ in R["top_features"][:5]]
bullets(s, ["Top drivers: " + ", ".join(tf),
            ("AQI today / recent lags rank among the top drivers \u2192 strong temporal persistence"
             if any(f.startswith("AQI") for f in tf[:3]) else "Pollutant readings outrank AQI history in importance"),
            "Particulate features (PM2.5, PM10) and rolling means: " +
            ("present in the top 10" if any(f.startswith("PM") for f, _ in R["top_features"]) else "not in the top 10"),
            "Check calendar / city features in the SHAP plot for seasonal and local effects"],
        7.9, 1.6, 5.0, 5, 17)

# 14 Findings
s = slide("Key Findings & Limitations", "Discussion")
bullets(s, [f"{b_reg} gave the best regression result (RMSE {bm['RMSE']:.1f}, R\u00b2 {bm['R2']:.2f}), "
            f"{gain:.1f}% better than simply repeating today's AQI",
            f"Best classifier: {b_clf} (macro-F1 {R['clf'][b_clf]['F1_macro']:.2f})",
            "Most influential features (" + R["interp_method"] + "): " + ", ".join(f for f, _ in R["top_features"][:3]),
            "Limitations: no meteorological data (wind, temperature, humidity), many missing readings, "
            "uneven station coverage, COVID-19 shift in the test window",
            "Rare 'Severe' days are hard to predict because of class imbalance"], 0.6, 1.6, 12, 5, 19)

# 15 Conclusion
s = slide("Conclusion & Future Work", "Wrap-up")
bullets(s, ["A complete KDD pipeline turned messy CPCB data into a working next-day AQI forecaster",
            ("Tree-based ensembles outperformed the linear baselines, indicating non-linear pollutant dynamics"
             if b_reg in ("Random Forest", "XGBoost") else
             f"{b_reg} performed best; the linear baselines remain strong because next-day AQI is highly autocorrelated"),
            "Chronological splitting and train-only preprocessing give an honest estimate of future performance",
            "Future work: add weather and fire-count data, LSTM / temporal models, multi-day horizons, "
            "per-city models, and a live health-advisory dashboard"], 0.6, 1.6, 12, 5, 20)

# 16 Thanks
s = prs.slides.add_slide(BLANK); page["n"] += 1
rect(s, 0, 0, 13.333, 7.5, NAVY)
text(s, "Thank you", 0.8, 2.4, 11.5, 1.2, 54, True, WHITE)
text(s, AUTHOR_NAME, 0.8, 3.8, 11.5, 0.6, 24, False, RGBColor(0x9F, 0xD6, 0xD6))
text(s, "Questions?", 0.8, 4.5, 11.5, 0.6, 20, False, RGBColor(0xC8, 0xD3, 0xDF))

out_pptx = os.path.join(OUT_DIR, "AQI_Prediction_KDD.pptx")
prs.save(out_pptx)
print("Saved:", out_pptx)

In [ ]:
# Download the finished presentation + results (Colab) or show where they are saved
try:
    from google.colab import files
    files.download(out_pptx); files.download(os.path.join(OUT_DIR, "results.json"))
except Exception:
    print("Saved in folder:", OUT_DIR)